In [1]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


In [2]:
%%writefile kernel.h
#pragma once // Empêche l'inclusion multiple de ce fichier d'en-tête

#include <iostream>
#include <vector>
#include <cmath>
#include <cfloat>

// ============================================================================
// 1. HYPERPARAMÈTRES DU DIFFERENTIAL EVOLUTION (DE)
// ============================================================================
#define DE_F  0.5f    // Facteur d'échelle / mutation (contrôle l'amplitude de l'écart)
#define DE_CR 0.8f    // Taux de croisement / Crossover (probabilité de hériter du mutant)

// ============================================================================
// 2. BORNES DE L'ESPACE DE RECHERCHE (Search Space)
// ============================================================================
#define LOWER_BOUND -5.12f  // Borne inférieure de l'espace de recherche
#define UPPER_BOUND  5.12f  // Borne supérieure de l'espace de recherche

// ============================================================================
// 3. CHOIX DE LA FONCTION OBJECTIF (BENCHMARK)
// Décommenter UNE SEULE ligne à la fois selon la fonction à tester
// ============================================================================
#define USE_RASTRIGIN
//#define USE_ROSENBROCK
//#define USE_SPHERE
//#define USE_GRIEWANK
//#define USE_LEVY

// ============================================================================
// 4. PROTOTYPES DES FONCTIONS UTILITAIRES ET CPU
// ============================================================================
// Génère un nombre flottant aléatoire uniforme entre 0.0 et 1.0
float getRandom();

// Génère un nombre flottant aléatoire uniforme entre min et max
float getRandomClamped(float min, float max);

// Calcule la valeur de la fonction objectif sur le processeur (CPU)
float host_fitness_function(const float* x, int dim);

// Exécute l'algorithme complet d'Évolution Différentielle en séquentiel sur CPU
float cpu_de(float* population, float* next_pop, float* fitness, 
             int pop_size, int dim, int max_iter);

// ============================================================================
// 5. PROTOTYPE DU MOTEUR GPU (CUDA)
// extern "C" permet la liaison entre la compilation C++ et le compilateur CUDA (NVCC)
// ============================================================================
extern "C" float cuda_de(float* h_population, int pop_size, int dim, int max_iter);

Writing kernel.h


Étape 1 : Le fichier kernel.h (L'en-tête / Le Header)
Rôle principal de kernel.h
Ce fichier est la carte d'identité du projet. Il sert de pont entre le C++ classique (CPU) et CUDA (GPU).

Configuration globale : C'est ici que tu définis les hyperparamètres du Differential Evolution (DE_F, DE_CR) et les bornes du problème.

Choix de la fonction : C'est le seul endroit où tu interviens pour choisir quelle fonction mathématique exécuter en décommentant sa ligne.

Déclarations : Il annonce au compilateur quelles fonctions existent dans kernel.cpp et kernel.cu pour que main.cpp puisse les appeler.

In [3]:
%%writefile kernel.cpp
#include "kernel.h"
#include <cstdlib>
#include <cmath>
#include <algorithm>

// ============================================================================
// 1. GÉNÉRATEURS DE NOMBRES ALÉATOIRES SUR CPU
// ============================================================================
float getRandom() {
    return static_cast<float>(rand()) / static_cast<float>(RAND_MAX);
}

float getRandomClamped(float min, float max) {
    return min + getRandom() * (max - min);
}

// ============================================================================
// 2. ÉVALUATION MATHÉMATIQUE SUR CPU (SÉQUENTIEL)
// ============================================================================
float host_fitness_function(const float* x, int dim) {
    float res = 0.0f;

#if defined(USE_RASTRIGIN)
    // Fonction Rastrigin : Multimodale avec de nombreux minima locaux
    for (int i = 0; i < dim; i++) {
        float zi = x[i];
        res += zi * zi - 10.0f * cosf(2.0f * M_PI * zi) + 10.0f;
    }

#elif defined(USE_ROSENBROCK)
    // Fonction Rosenbrock : Vallée non convexe ("banane de Rosenbrock")
    for (int i = 0; i < dim - 1; i++) {
        float zi = x[i];
        float zip1 = x[i + 1];
        res += 100.0f * powf(zi * zi - zip1, 2.0f) + powf(zi - 1.0f, 2.0f);
    }

#elif defined(USE_SPHERE)
    // Fonction Sphere : Unimodale, simple et convexe (Baseline)
    for (int i = 0; i < dim; i++) {
        res += x[i] * x[i];
    }

#elif defined(USE_GRIEWANK)
    // Fonction Griewank : Très multimodale avec de nombreux minima locaux
    float somme = 0.0f;
    float produit = 1.0f;
    for (int i = 0; i < dim; i++) {
        somme += (x[i] * x[i]) / 4000.0f;
        produit *= cosf(x[i] / sqrtf(i + 1));
    }
    res = somme - produit + 1.0f;

#elif defined(USE_LEVY)
    // Fonction Levy : Relief complexe avec de nombreux pièges
    float w1 = 1.0f + (x[0] - 1.0f) / 4.0f;
    float wn = 1.0f + (x[dim - 1] - 1.0f) / 4.0f;
    res = powf(sinf(M_PI * w1), 2.0f);
    for (int i = 0; i < dim - 1; i++) {
        float wi = 1.0f + (x[i] - 1.0f) / 4.0f;
        res += powf(wi - 1.0f, 2.0f) * (1.0f + 10.0f * powf(sinf(M_PI * wi + 1.0f), 2.0f));
    }
    res += powf(wn - 1.0f, 2.0f) * (1.0f + powf(sinf(2.0f * M_PI * wn), 2.0f));
#endif

    return res;
}

// ============================================================================
// 3. BOUCLE PRINCIPALE DU DIFFERENTIAL EVOLUTION SUR CPU
// ============================================================================
float cpu_de(float* population, float* next_pop, float* fitness, 
             int pop_size, int dim, int max_iter) {
    
    // Évaluation initiale de toute la population
    for (int i = 0; i < pop_size; i++) {
        fitness[i] = host_fitness_function(&population[i * dim], dim);
    }

    float best_fitness = FLT_MAX;
    
    // Boucle des générations
    for (int iter = 0; iter < max_iter; iter++) {
        // Parcours séquentiel de chaque individu
        for (int i = 0; i < pop_size; i++) {
            
            // Étape A : Sélection de 3 individus r1 != r2 != r3 != i
            int a, b, c;
            do { a = rand() % pop_size; } while (a == i);
            do { b = rand() % pop_size; } while (b == i || b == a);
            do { c = rand() % pop_size; } while (c == i || c == a || c == b);

            int j_rand = rand() % dim; // Assure qu'au moins une dimension est mutée
            float trial[128];          // Vecteur candidat temporaire

            // Étape B : Mutation et Croisement (Crossover)
            for (int d = 0; d < dim; d++) {
                if (getRandom() < DE_CR || d == j_rand) {
                    // Formule du vecteur mutant : V = X_a + F * (X_b - X_c)
                    trial[d] = population[a * dim + d] + DE_F * (population[b * dim + d] - population[c * dim + d]);
                } else {
                    // Conservation du membre original
                    trial[d] = population[i * dim + d];
                }
            }

            // Étape C : Évaluation du nouveau candidat
            float f_trial = host_fitness_function(trial, dim);

            // Étape D : Sélection "Gourmande" (Greedy Selection)
            if (f_trial <= fitness[i]) {
                fitness[i] = f_trial; // Le candidat est meilleur, on le conserve
                for (int d = 0; d < dim; d++) {
                    next_pop[i * dim + d] = trial[d];
                }
            } else {
                // Le candidat est moins bon, on garde l'ancien
                for (int d = 0; d < dim; d++) {
                    next_pop[i * dim + d] = population[i * dim + d];
                }
            }

            // Mise à jour de la meilleure solution globale
            if (fitness[i] < best_fitness) {
                best_fitness = fitness[i];
            }
        }

        // Passage à la nouvelle génération
        for (int k = 0; k < pop_size * dim; k++) {
            population[k] = next_pop[k];
        }
    }

    return best_fitness;
}

Writing kernel.cpp


Étape 2 : Le fichier kernel.cpp (Le moteur CPU)
Rôle principal de kernel.cpp
Ce fichier contient la version séquentielle (CPU) de ton algorithme.

Il implémente les 5 fonctions mathématiques (host_fitness_function) pour le processeur.

Il contient la boucle principale de l'Évolution Différentielle qui traite un individu après l'autre sur le CPU.

Son utilité majeure : Servir de point de référence (baseline). Sans ce fichier, impossible de prouver le gain de temps apporté par la carte graphique !

In [4]:
%%writefile kernel.cu
#include "kernel.h"
#include <cuda_runtime.h>
#include <curand_kernel.h>
#include <cfloat>
#include <iostream>

// ============================================================================
// 1. ÉVALUATION MATHÉMATIQUE SUR GPU (__device__)
// Exécutée par chaque thread GPU pour calculer la valeur de fitness
// ============================================================================
__device__ float device_fitness_function(const float* x, int dim) {
    float res = 0.0f;

#if defined(USE_RASTRIGIN)
    // Fonction Rastrigin parallèle
    for (int i = 0; i < dim; i++) {
        float zi = x[i];
        res += zi * zi - 10.0f * cosf(2.0f * M_PI * zi) + 10.0f;
    }

#elif defined(USE_ROSENBROCK)
    // Fonction Rosenbrock parallèle
    for (int i = 0; i < dim - 1; i++) {
        float zi = x[i];
        float zip1 = x[i + 1];
        res += 100.0f * powf(zi * zi - zip1, 2.0f) + powf(zi - 1.0f, 2.0f);
    }

#elif defined(USE_SPHERE)
    // Fonction Sphere parallèle
    for (int i = 0; i < dim; i++) {
        res += x[i] * x[i];
    }

#elif defined(USE_GRIEWANK)
    // Fonction Griewank parallèle
    float somme = 0.0f;
    float produit = 1.0f;
    for (int i = 0; i < dim; i++) {
        somme += (x[i] * x[i]) / 4000.0f;
        produit *= cosf(x[i] / sqrtf(i + 1));
    }
    res = somme - produit + 1.0f;

#elif defined(USE_LEVY)
    // Fonction Levy parallèle
    float w1 = 1.0f + (x[0] - 1.0f) / 4.0f;
    float wn = 1.0f + (x[dim - 1] - 1.0f) / 4.0f;
    res = powf(sinf(M_PI * w1), 2.0f);
    for (int i = 0; i < dim - 1; i++) {
        float wi = 1.0f + (x[i] - 1.0f) / 4.0f;
        res += powf(wi - 1.0f, 2.0f) * (1.0f + 10.0f * powf(sinf(M_PI * wi + 1.0f), 2.0f));
    }
    res += powf(wn - 1.0f, 2.0f) * (1.0f + powf(sinf(2.0f * M_PI * wn), 2.0f));
#endif

    return res;
}

// ============================================================================
// 2. INITIALISATION DES ÉTATS ALÉATOIRES SUR GPU (__global__)
// Chaque thread initialise son propre état cuRAND avec une graine (seed)
// ============================================================================
__global__ void setup_kernel(curandState *state, unsigned long seed, int pop_size) {
    int id = threadIdx.x + blockIdx.x * blockDim.x;
    if (id < pop_size) {
        curand_init(seed, id, 0, &state[id]);
    }
}

// ============================================================================
// 3. KERNEL CUDA PRINCIPAL : 1 ÉTAPE DE DE PAR THREAD (__global__)
// Chaque thread gère UN individu de la population en parallèle
// ============================================================================
__global__ void kernel_de_step(float* d_population, float* d_next_pop, float* d_fitness, 
                               curandState* state, int pop_size, int dim) {
    
    int id = threadIdx.x + blockIdx.x * blockDim.x;
    if (id >= pop_size) return; // Sécurité hors bornes

    curandState localState = state[id];

    // Tirage aléatoire de 3 individus distincts r1 != r2 != r3 != id
    int a, b, c;
    do { a = curand(&localState) % pop_size; } while (a == id);
    do { b = curand(&localState) % pop_size; } while (b == id || b == a);
    do { c = curand(&localState) % pop_size; } while (c == id || c == a || c == b);

    int j_rand = curand(&localState) % dim;
    float trial[128]; // Vecteur mutant temporaire dans la mémoire locale du thread

    // Mutation & Croisement (Crossover)
    for (int d = 0; d < dim; d++) {
        float rand_val = curand_uniform(&localState);
        if (rand_val < DE_CR || d == j_rand) {
            trial[d] = d_population[a * dim + d] + DE_F * (d_population[b * dim + d] - d_population[c * dim + d]);
        } else {
            trial[d] = d_population[id * dim + d];
        }
    }

    state[id] = localState; // Sauvegarde de l'état aléatoire

    // Évaluation du candidat sur GPU
    float f_trial = device_fitness_function(trial, dim);

    // Sélection Gourmande
    if (f_trial <= d_fitness[id]) {
        d_fitness[id] = f_trial; // Conserve la nouvelle valeur
        for (int d = 0; d < dim; d++) {
            d_next_pop[id * dim + d] = trial[d];
        }
    } else {
        for (int d = 0; d < dim; d++) {
            d_next_pop[id * dim + d] = d_population[id * dim + d];
        }
    }
}

// ============================================================================
// 4. FONCTION D'ORCHESTRATION GPU (Appelée par le C++ / main.cpp)
// ============================================================================
extern "C" float cuda_de(float* h_population, int pop_size, int dim, int max_iter) {
    
    size_t pop_bytes = pop_size * dim * sizeof(float);
    size_t fit_bytes = pop_size * sizeof(float);

    // Pointeurs vers la mémoire GPU (VRAM)
    float *d_population, *d_next_pop, *d_fitness;
    curandState *d_state;

    // Allocation VRAM
    cudaMalloc(&d_population, pop_bytes);
    cudaMalloc(&d_next_pop, pop_bytes);
    cudaMalloc(&d_fitness, fit_bytes);
    cudaMalloc(&d_state, pop_size * sizeof(curandState));

    // Copie de la population du CPU (Host) vers le GPU (Device)
    cudaMemcpy(d_population, h_population, pop_bytes, cudaMemcpyHostToDevice);

    // Initialisation du tableau de fitness
    float* h_fitness = new float[pop_size];
    for (int i = 0; i < pop_size; i++) {
        h_fitness[i] = FLT_MAX;
    }
    cudaMemcpy(d_fitness, h_fitness, fit_bytes, cudaMemcpyHostToDevice);

    // Configuration de la grille CUDA (Blocks & Threads)
    int threadsPerBlock = 64;
    int blocksPerGrid = (pop_size + threadsPerBlock - 1) / threadsPerBlock;

    // Initialisation de cuRAND
    setup_kernel<<<blocksPerGrid, threadsPerBlock>>>(d_state, 1234, pop_size);

    // Boucle des générations sur GPU
    for (int iter = 0; iter < max_iter; iter++) {
        kernel_de_step<<<blocksPerGrid, threadsPerBlock>>>(d_population, d_next_pop, d_fitness, d_state, pop_size, dim);
        // Mise à jour de la population pour la génération suivante
        cudaMemcpy(d_population, d_next_pop, pop_bytes, cudaMemcpyDeviceToDevice);
    }

    // Copie des résultats GPU vers CPU
    cudaMemcpy(h_fitness, d_fitness, fit_bytes, cudaMemcpyDeviceToHost);

    // Recherche de la meilleure fitness finale
    float best_fitness = FLT_MAX;
    for (int i = 0; i < pop_size; i++) {
        if (h_fitness[i] < best_fitness) {
            best_fitness = h_fitness[i];
        }
    }

    // Nettoyage de la mémoire VRAM et RAM
    cudaFree(d_population);
    cudaFree(d_next_pop);
    cudaFree(d_fitness);
    cudaFree(d_state);
    delete[] h_fitness;

    return best_fitness;
}

Writing kernel.cu


Étape 3 : Le fichier kernel.cu (Le moteur GPU / CUDA)Rôle principal de kernel.cuC'est le cœur parallèle de ton application sur carte graphique.Gestion VRAM : Il alloue la mémoire sur le GPU (cudaMalloc) et gère les transferts de données CPU $\rightarrow$ GPU $\rightarrow$ CPU (cudaMemcpy).Calcul Parallèle : Il distribue le traitement de la population sur des centaines de threads GPU en parallèle via le kernel kernel_de_step.RNG GPU : Il utilise la bibliothèque cuRAND (curandState) pour que chaque thread génère ses propres nombres aléatoires en parallèle sans bloquer les autres.

In [5]:
%%writefile main.cpp
#include "kernel.h"
#include <iostream>
#include <vector>
#include <chrono>
#include <iomanip>
#include <cstdlib>

int main(int argc, char** argv) {
    // Parameters par défaut
    int dim = 30;         // Dimension du problème (30 variables)
    int pop_size = 256;   // Taille de la population (256 individus)
    int max_iter = 2000;  // Nombre de générations (2000 itérations)

    // Possibilité de passer les paramètres en ligne de commande
    if (argc >= 3) {
        dim = std::stoi(argv[1]);
        pop_size = std::stoi(argv[2]);
    }

    std::cout << "==================================================" << std::endl;
    std::cout << "  Algorithme d'Evolution Differentielle (DE)" << std::endl;
    std::cout << "  Population : " << pop_size << " | Dimension : " << dim << std::endl;
    std::cout << "  Generations: " << max_iter << std::endl;
    std::cout << "==================================================" << std::endl;

    // Allocation mémoire des structures CPU
    std::vector<float> population(pop_size * dim);
    std::vector<float> next_pop(pop_size * dim);
    std::vector<float> fitness(pop_size);

    // Graine fixe pour garantir que CPU et GPU partent de la MÊME population
    srand(1234);
    for (int i = 0; i < pop_size * dim; i++) {
        population[i] = getRandomClamped(LOWER_BOUND, UPPER_BOUND);
    }

    // Copie de sauvegarde pour le GPU
    std::vector<float> gpu_population = population;

    // ========================================================================
    // 1. CALCUL CPU
    // ========================================================================
    std::cout << "\n[1] Lancement du calcul CPU..." << std::endl;
    auto start_cpu = std::chrono::high_resolution_clock::now();
    
    float best_fitness_cpu = cpu_de(population.data(), next_pop.data(), fitness.data(), 
                                    pop_size, dim, max_iter);
    
    auto end_cpu = std::chrono::high_resolution_clock::now();
    std::chrono::duration<double, std::milli> duration_cpu = end_cpu - start_cpu;

    // ========================================================================
    // 2. CALCUL GPU (CUDA)
    // ========================================================================
    std::cout << "[2] Lancement du calcul GPU (CUDA)..." << std::endl;
    auto start_gpu = std::chrono::high_resolution_clock::now();
    
    float best_fitness_gpu = cuda_de(gpu_population.data(), pop_size, dim, max_iter);
    
    auto end_gpu = std::chrono::high_resolution_clock::now();
    std::chrono::duration<double, std::milli> duration_gpu = end_gpu - start_gpu;

    // ========================================================================
    // 3. AFFICHAGE DES RÉSULTATS COMPARATIFS
    // ========================================================================
    double speedup = duration_cpu.count() / duration_gpu.count();

    std::cout << "\n==================================================" << std::endl;
    std::cout << std::left << std::setw(10) << "Mode" 
              << std::setw(18) << "Temps (ms)" 
              << "Meilleure Fitness" << std::endl;
    std::cout << "--------------------------------------------------" << std::endl;
    std::cout << std::left << std::setw(10) << "CPU" 
              << std::setw(18) << duration_cpu.count() 
              << best_fitness_cpu << std::endl;
    std::cout << std::left << std::setw(10) << "GPU" 
              << std::setw(18) << duration_gpu.count() 
              << best_fitness_gpu << std::endl;
    std::cout << "==================================================" << std::endl;
    std::cout << "   >>> ACCELERATION (SPEEDUP) : " << std::fixed << std::setprecision(2) 
              << speedup << "x <<<" << std::endl;
    std::cout << "==================================================" << std::endl;

    return 0;
}

Writing main.cpp


Étape 4 : Le fichier main.cpp (Le programme principal)Rôle principal de main.cppC'est le chef d'orchestre du projet.Il initialise la population de départ de manière aléatoire.Il exécute et mesure le temps du CPU (cpu_de).Il exécute et mesure le temps du GPU (cuda_de).Il compare les deux résultats et affiche le Speedup ($\frac{\text{Temps CPU}}{\text{Temps GPU}}$).

In [6]:
!nvcc -O3 main.cpp kernel.cpp kernel.cu -o de_app && ./de_app

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
  Algorithme d'Evolution Differentielle (DE)
  Population : 256 | Dimension : 30
  Generations: 2000

[1] Lancement du calcul CPU...
[2] Lancement du calcul GPU (CUDA)...

Mode      Temps (ms)        Meilleure Fitness
--------------------------------------------------
CPU       523.98            178.443
GPU       849.003           168.767
   >>> ACCELERATION (SPEEDUP) : 0.62x <<<
